In [ ]:
list.of.packages = c("tidyverse", "bigrquery")
install.packages(list.of.packages)

library(tidyverse)
library(bigrquery)

# The below queries represent dataset AllOfUsControlledTierDatasetv9 in cohort hhtepi at 20260827_062517.

# If EXPORT_BUCKET is not set in your environment, uncomment and update the line below
# with your actual Cloud Storage bucket name from the Resources tab in your workspace.
Sys.setenv(EXPORT_BUCKET = "gs://hht-epi-bucket-wb-lucent-greens-2488")

# Domain person
person_sql = r"(

    SELECT
        st.birth_datetime,
        st.ethnicity_concept_id,
        dt0.concept_name AS T_DISP_ethnicity,
        st.gender_concept_id,
        dt1.concept_name AS T_DISP_gender,
        st.person_id,
        st.race_concept_id,
        dt2.concept_name AS T_DISP_race,
        st.self_reported_category_concept_id,
        dt3.concept_name AS T_DISP_self_reported_category,
        st.sex_at_birth_concept_id,
        dt4.concept_name AS T_DISP_sex_at_birth 
    FROM
        `wb-silky-artichoke-2408.C2025Q4R6`.person AS st 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt0 
            ON dt0.concept_id = st.ethnicity_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt1 
            ON dt1.concept_id = st.gender_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt2 
            ON dt2.concept_id = st.race_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt3 
            ON dt3.concept_id = st.self_reported_category_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt4 
            ON dt4.concept_id = st.sex_at_birth_concept_id 
    WHERE
        st.person_id IN (SELECT
            id 
        FROM
            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_person 
        WHERE
            (id IN (SELECT
                person_id AS primary_id 
            FROM
                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_conditionOccurrence 
            WHERE
                condition_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
                WHERE
                    ancestor = 313504 
                UNION
                ALL SELECT
                    313504))) 
                OR ((id IN (SELECT
                    flattened_person_id 
                FROM
                    (SELECT
                        DISTINCT vid, flattened_person_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                    CROSS JOIN
                        UNNEST(ARRAY(SELECT
                            element 
                        FROM
                            UNNEST(person_ids.list))) AS flattened_person_id 
                    JOIN
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                            ON sp.person_id = flattened_person_id 
                    WHERE
                        vid IN (SELECT
                            id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                        WHERE
                            (id IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                            WHERE
                                gene = 'ENG')) 
                            AND (EXISTS (SELECT
                                * 
                            FROM
                                UNNEST(clinvar_significance) AS flattened 
                            WHERE
                                flattened IN ('likely pathogenic', 'pathogenic'))))))) 
                OR (id IN (SELECT
                    flattened_person_id 
                FROM
                    (SELECT
                        DISTINCT vid, flattened_person_id 
                    FROM
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                    CROSS JOIN
                        UNNEST(ARRAY(SELECT
                            element 
                        FROM
                            UNNEST(person_ids.list))) AS flattened_person_id 
                    JOIN
                        `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                            ON sp.person_id = flattened_person_id 
                    WHERE
                        vid IN (SELECT
                            id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                        WHERE
                            (id IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                            WHERE
                                gene = 'ACVRL1')) 
                            AND (EXISTS (SELECT
                                * 
                            FROM
                                UNNEST(clinvar_significance) AS flattened 
                            WHERE
                                flattened IN ('likely pathogenic', 'pathogenic'))))))))))"

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
person_path <- file.path(
  "gs://hht-epi-bucket-wb-lucent-greens-2488",
  "bq_exports",
  "person",
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "person",
  "person_*.csv")
message(str_glue('The data will be written to {person_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query("wb-silky-artichoke-2408.C2025Q4R6_index_061026", person_sql, billing = "wb-lucent-greens-2488"),
  person_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {person_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

person_df <- read_bq_export_from_workspace_bucket(person_path)
dim(person_df)
head(person_df, 5)




# Domain zipcodeSocioeconomic
zipcodeSocioeconomic_sql = r"(

    SELECT
        american_community_survey_year,
        assisted_income,
        deprivation_index,
        high_school_education,
        id,
        median_income,
        no_health_insurance,
        observation_datetime_str,
        observation_datetime_utc,
        person_id,
        poverty,
        vacant_housing,
        zip_code 
    FROM
        `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_zipcodeSocioeconomic 
    WHERE
        (
            person_id IN (SELECT
                person_id AS primary_id 
            FROM
                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_conditionOccurrence 
            WHERE
                condition_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
                WHERE
                    ancestor = 313504 
                UNION
                ALL SELECT
                    313504))
            ) 
            OR (
                (
                    person_id IN (SELECT
                        flattened_person_id 
                    FROM
                        (SELECT
                            DISTINCT vid, flattened_person_id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                        CROSS JOIN
                            UNNEST(ARRAY(SELECT
                                element 
                            FROM
                                UNNEST(person_ids.list))) AS flattened_person_id 
                        JOIN
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                ON sp.person_id = flattened_person_id 
                        WHERE
                            vid IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                            WHERE
                                (id IN (SELECT
                                    id 
                                FROM
                                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                                WHERE
                                    gene = 'ENG')) 
                                AND (EXISTS (SELECT
                                    * 
                                FROM
                                    UNNEST(clinvar_significance) AS flattened 
                                WHERE
                                    flattened IN ('likely pathogenic', 'pathogenic'))))))
                ) 
                OR (
                    person_id IN (SELECT
                        flattened_person_id 
                    FROM
                        (SELECT
                            DISTINCT vid, flattened_person_id 
                        FROM
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_variant_to_person` 
                        CROSS JOIN
                            UNNEST(ARRAY(SELECT
                                element 
                            FROM
                                UNNEST(person_ids.list))) AS flattened_person_id 
                        JOIN
                            `wb-silky-artichoke-2408.C2025Q4R6.cb_search_person` AS sp  
                                ON sp.person_id = flattened_person_id 
                        WHERE
                            vid IN (SELECT
                                id 
                            FROM
                                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_variant 
                            WHERE
                                (id IN (SELECT
                                    id 
                                FROM
                                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_variant_gene 
                                WHERE
                                    gene = 'ACVRL1')) 
                                AND (EXISTS (SELECT
                                    * 
                                FROM
                                    UNNEST(clinvar_significance) AS flattened 
                                WHERE
                                    flattened IN ('likely pathogenic', 'pathogenic'))))))
                )
            ))"

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
zipcodeSocioeconomic_path <- file.path(
  "gs://hht-epi-bucket-wb-lucent-greens-2488",
  "bq_exports",
  "zipcodeSocioeconomic",
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "zipcodeSocioeconomic",
  "zipcodeSocioeconomic_*.csv")
message(str_glue('The data will be written to {zipcodeSocioeconomic_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query("wb-silky-artichoke-2408.C2025Q4R6_index_061026", zipcodeSocioeconomic_sql, billing = "wb-lucent-greens-2488"),
  zipcodeSocioeconomic_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {zipcodeSocioeconomic_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

zipcodeSocioeconomic_df <- read_bq_export_from_workspace_bucket(zipcodeSocioeconomic_path)
dim(zipcodeSocioeconomic_df)
head(zipcodeSocioeconomic_df, 5)


Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done

